# 2025-26 Project - Foundations of Computer Science

**Project by:**
- Herandez, Nicole Winy </br>
- Ortega, Azriel Matthew </br>
- Tibayan, Meryann Joy

In [1]:
# import libraries
import pandas as pd
import os
import json

### 1. Create a single dataframe with the concatenation of all input csv files, adding a column called country

In [2]:
# read csv files with read_csv function from pandas
videos_folder = "videos"

df_cavideos = pd.read_csv(f"{videos_folder}/CAvideos.csv")
df_devideos = pd.read_csv(f"{videos_folder}/DEvideos.csv")
df_frvideos = pd.read_csv(f"{videos_folder}/FRvideos.csv")
df_gbvideos = pd.read_csv(f"{videos_folder}/GBvideos.csv")
df_invideos = pd.read_csv(f"{videos_folder}/INvideos.csv")
df_usvideos = pd.read_csv(f"{videos_folder}/USvideos.csv")

# After experimenting, it seems that using default encoding(utf-8) and using encoding errors
# to skip some bytes that cant be read is better than finding a specific encoding for each language
# However, this must be researched more for example: If this only a computer specific problem, or 
# there is a way to read everything properly without having to use encoding errors

df_jpvideos = pd.read_csv(f"{videos_folder}/JPvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_krvideos = pd.read_csv(f"{videos_folder}/KRvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_mxvideos = pd.read_csv(f"{videos_folder}/MXvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_ruvideos = pd.read_csv(f"{videos_folder}/RUvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")

They say it is hard to determine the correct encoding for each text file, especially if it is not utf-8 </br>
Reference https://community.dataquest.io/t/how-do-we-know-the-encoding-to-use-with-pd-read-csv-chardet-doesnt-help/467614/4 </br>

You have to do trial and error with all of the encodings to check which one works best. As for my testing 11/20/2025, utf-8 is best for all with the encoding_errors set to replace.

Here is the docs for checking all encodings : https://docs.python.org/3/library/codecs.html#standard-encodings

In [ ]:
# Efficient code version (with concatenation process

# Import all csv files, read and compile them into one dataframe
# With the addition of adding a new column 'country' to determine from what csv file it came from

dataframe_lists = []
for file in os.listdir(videos_folder):
    if file[9:] == "csv":
        print(f"Reading {file}")
        df_temp = pd.read_csv(f"{videos_folder}/{file}", encoding_errors="replace")
        df_temp["country"] = file[:2] # add country column
        dataframe_lists.append(df_temp)

df_videos = pd.concat(dataframe_lists) # concatenate everything into one dataframe
df_videos = df_videos.reset_index(drop=True) # reset index

Reading MXvideos.csv
Reading DEvideos.csv
Reading FRvideos.csv
Reading INvideos.csv
Reading RUvideos.csv
Reading JPvideos.csv


In [ ]:
df_videos

### 2. Extract all videos that have no tag.

In [ ]:
no_tag_videos = df_videos[df_videos["tags"] == "[none]"]
no_tag_videos #all videos

In [ ]:
no_tag_videos_unici = no_tag_videos.drop_duplicates(subset=["video_id"])
no_tag_videos_unici #unique videos by video_id #there are 23992 unique videos that have no tags

### 3. For each channel, determine the total number of views

In [ ]:
total_views_per_channel = df_videos.groupby("channel_title")["views"].sum().sort_values(ascending=False)
total_views_per_channel 

### 4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

In [ ]:
df_videos_disabled = df_videos[df_videos["comments_disabled"] | df_videos["ratings_disabled"] | df_videos["video_error_or_removed"]]
df_videos_disabled

In [ ]:
df_videos = df_videos.drop(df_videos_disabled.index)
df_videos

### 5. Add a like_ratio column storing the ratio between the number of likes and of dislikes

In [ ]:
import numpy as np
df_videos["like_ratio"] = df_videos["likes"] / df_videos["dislikes"].replace(0, np.nan)
df_videos[["title", "likes", "dislikes", "like_ratio"]].head(10)

In [ ]:
# if needed: how many rows have like_ratio = NaN
num_nan_ratio = df_videos["like_ratio"].isna().sum()
print("Righe con like_ratio NaN:", num_nan_ratio)

# rows with like_ratio NaN
ratio_nan_rows = df_videos[df_videos["like_ratio"].isna()]
ratio_nan_rows.head(5)


### 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

In [ ]:
pt = pd.to_datetime(df_videos["publish_time"], utc=True, errors="coerce")
start = pt.dt.floor("10min")
df_videos["publish_interval"] = (
    start.dt.strftime("%H:%M") + " - " +
    (start + pd.Timedelta(minutes=10)).dt.strftime("%H:%M")
)

df_videos[["publish_time", "publish_interval"]]

### 7. For each interval, determine the number of videos, average number of likes and of dislikes.

In [ ]:
# Unique Videos

interval_stats = df_videos.groupby("publish_interval").agg(
    number_of_videos = ("video_id", "nunique"),
    average_likes = ("likes", "mean"),
    average_dislikes = ("dislikes", "mean")
)

interval_stats.sort_index().head(50)

In [ ]:
# By Rows
interval_stats = df_videos.groupby("publish_interval").agg(
    number_of_videos = ("video_id", "count"),
    average_likes = ("likes", "mean"),
    average_dislikes = ("dislikes", "mean")
)

interval_stats.sort_index().head(50)

### 8. For each tag, determine the number of videos. Notice that tags contains a string with several tags

Setup tag column in our dataframe first <br>
Split the string into array with '|' as the separator <br>

Explode dataframe so that each row wiill have one tag, it will duplicate rows with many tags. This is the approach because it is useful when grouping by tags.

In [ ]:
df_videos_tags = df_videos.copy()

In [ ]:
df_videos_tags["tags"] = df_videos_tags["tags"].str.split('|')

In [ ]:
df_videos_tags = df_videos_tags[["video_id", "tags", "like_ratio", "country", "trending_date"]]

In [ ]:
df_videos_tags_explode = df_videos_tags.explode('tags') 

In [ ]:
# clean dataset by removing all extra characters such as double quotations (")
df_videos_tags_explode["tags"] = df_videos_tags_explode["tags"].str.replace('"', '').str.strip()

In [ ]:
df_videos_tags_explode = df_videos_tags_explode[df_videos_tags_explode['tags'].str.strip() != ""]

In [ ]:
# Videos with no tags are excluded
df_videos_tags_explode.groupby(by="tags")['video_id'].nunique()

In [ ]:
# Number of videos with no tags (if needed)
df_videos_tags_explode[df_videos_tags_explode["tags"] == "[none]"]["video_id"].nunique()

### 9. Find the tags with the largest number of videos

In [ ]:
df_videos_tags_explode[df_videos_tags_explode["tags"] != "[none]"].groupby(by="tags")['video_id'].nunique().sort_values(ascending=False).head(1)

### 10. For each (tag, country) pair, compute average ratio likes/dislikes

In [ ]:
df_videos_tags_only = df_videos_tags_explode[(df_videos_tags_explode["tags"] != "[none]")]

In [ ]:
pd.DataFrame(df_videos_tags_only.groupby(by=["tags","country"])["like_ratio"].mean())

### 11. For each (trending_date, country) pair, the video with the largest number of views

In [ ]:
max_views = df_videos.groupby(["trending_date", "country"])["views"].transform("max")
max_views

In [ ]:
max_views = df_videos.groupby(["trending_date", "country"])["views"].transform("max")
df_videos_group = df_videos[df_videos["views"] == max_views]

In [ ]:
# Sort columns, to display Country, Trending Date and then Views First.
priority_columns = ["trending_date","country", "views"]
other_columns = [col for col in df_videos.columns if col not in priority_columns]

df_videos_group[priority_columns + other_columns]

### 12. Divide trending_date into three columns: year, month, day

In [ ]:
# Split the 'trending_date' column
# The expand=True parameter turns the result into a DataFrame
data_split = df_videos['trending_date'].str.split('.', expand=True)

# Assign New Columns
# Index 0 = Year (YY) -> add '20' to get '2017'
df_videos['year'] = '20' + data_split[0].astype(str)
# Index 1 = Day (DD)
df_videos['day'] = data_split[1]
# Index 2 = Month (MM)
df_videos['month'] = data_split[2]

print(df_videos[['video_id','trending_date', 'year', 'month', 'day']].head())

In [ ]:
# Convert to datetime format
# Use format YYYY-MM-DD
df_videos['trending_date'] = pd.to_datetime(df_videos['trending_date'], format='%y.%d.%m')

# Extract the components
df_videos['year'] = df_videos['trending_date'].dt.year
df_videos['month'] = df_videos['trending_date'].dt.month
df_videos['day'] = df_videos['trending_date'].dt.day

print(df_videos[['trending_date', 'year', 'month', 'day']].head())

### 13. For each (month, country) pair, the video with the largest number of views

In [ ]:
# Group by 'month' and 'country'
# Find the index of the row with the highest 'views' value in each group
max_views_indices = df_videos.groupby(['month', 'country'])['views'].idxmax()

# 3. Use .loc to select the rows from the original DataFrame
df_max_views = df_videos.loc[max_views_indices]

print(df_max_views[['month', 'country', 'video_id','title', 'views']].head())

### 14. Read all json files with the video categories

In [ ]:
json_folder = 'categories'
df_list = []

for file in os.listdir(json_folder):
    if file.endswith(".json"):
        print(f"Reading {file}")
        
        with open(os.path.join(json_folder, file), 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Correct normalization for YouTube categories
        df_temp = pd.json_normalize(data['items'])
        
        # Select only the useful columns
        df_temp = df_temp[['id', 'snippet.title', 'snippet.assignable']]
        
        # Rename for consistency with the video dataframe
        df_temp.columns = ['category_id', 'category_name', 'assignable']
        
        # Convert category_id to integer
        df_temp['category_id'] = df_temp['category_id'].astype(int)
        
        df_list.append(df_temp)

df_categories = pd.concat(df_list, ignore_index=True)
df_categories = df_categories.drop_duplicates(subset=['category_id']).reset_index(drop=True)

df_categories


### 15. For each country, determine how many videos have a category that is not assignable

In [ ]:
df_merged = df_videos.merge(df_categories, on='category_id', how='left')

In [ ]:
df_not_assignable = (df_merged[df_merged['assignable'] == False]
    .groupby('country')['video_id']
    .count()
    .reset_index(name='non_assignable_count')
)

# this counts number of rows that are not assignable
df_not_assignable

In [ ]:
# this counts number of unique videos that are not assignable
df_not_assignable_unique = (df_merged[df_merged['assignable'] == False]
    .groupby('country')['video_id']
    .nunique()
    .reset_index(name='non_assignable_count')
)

df_not_assignable_unique